In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from sklearn.metrics import roc_curve

# --- 1. The Academic MLP Architecture ---
class BioAcademicMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

# --- 2. Function to Calculate EER ---
def calculate_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return eer * 100

# --- 3. Feature Prep (Must perfectly match training) ---
def prepare_features(df):
    df = df.copy()
    
    # Safety cleanup
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    # Feature engineering
    if "meanF0" in df.columns and "stdevF0" in df.columns:
        df["f0_cov"] = df["stdevF0"] / (df["meanF0"] + 1e-6)

    # Log transforms
    jsh_cols = [
        "j_local", "j_abs", "j_rap", "j_ppq5", "j_ddp",
        "s_local", "s_db", "s_apq3", "s_apq5", "s_apq11", "s_dda"
    ]
    for col in jsh_cols:
        if col in df.columns:
            df[col] = np.log(np.clip(df[col].astype(float), 1e-6, None))

    feature_cols = ["meanF0", "f0_cov", "hnr"] + jsh_cols
    
    X = df[feature_cols].values.astype(np.float32)
    y = df["label"].values.astype(np.float32)
    
    return X, y

def main():
    # 1. Load the Scaler
    print("Loading scaler from training phase...")
    scaler = joblib.load(r"C:\Users\adith\Desktop\Projects\Audio Deepfake\bio_academic_scaler.pkl")

    # 2. Load the Model
    input_dim = 14
    model = BioAcademicMLP(input_dim)
    model.load_state_dict(torch.load(r"C:\Users\adith\Desktop\Projects\Audio Deepfake\bio_academic_mlp_weights.pt"))
    model.eval()
    
    # 3. Load the Evaluation Data
    print("Loading evaluation dataset (this may take a moment)...")
    eval_df = pd.read_csv(r"C:\Users\adith\Desktop\Projects\Audio Deepfake\eval_df_bio_academic.csv")
    
    # Separate into Clean (nocodec) and Compressed
    clean_df = eval_df[eval_df['codec'] == 'nocodec']
    compressed_df = eval_df[eval_df['codec'] != 'nocodec']
    
    datasets = {"Clean (No Codec)": clean_df, "Compressed": compressed_df}
    
    for condition, df in datasets.items():
        if len(df) == 0:
            print(f"\n--- Condition: {condition} Audio ---")
            print("No files found for this condition.")
            continue
            
        # Prep features (f0_cov, log transform, etc.)
        X_eval, y_eval = prepare_features(df)
        
        # Scale the features
        X_eval_scaled = scaler.transform(X_eval)
        X_tensor = torch.tensor(X_eval_scaled, dtype=torch.float32)
        
        # 4. Batched Inference (To prevent RAM exhaustion)
        batch_size = 5000
        all_scores = []
        
        with torch.no_grad():
            for i in range(0, len(X_tensor), batch_size):
                batch = X_tensor[i:i+batch_size]
                logits = model(batch)
                scores = torch.sigmoid(logits).numpy().flatten()
                all_scores.extend(scores)
                
        # Calculate and print EER
        eer = calculate_eer(y_eval, all_scores)
        print(f"\n--- Condition: {condition} Audio ---")
        print(f"Files tested: {len(X_eval)}")  # using X_eval len as some might have dropped
        print(f"Equal Error Rate (EER): {eer:.2f}%")

if __name__ == "__main__":
    main()

Loading scaler from training phase...
Loading evaluation dataset (this may take a moment)...

--- Condition: Clean (No Codec) Audio ---
Files tested: 39132
Equal Error Rate (EER): 28.91%

--- Condition: Compressed Audio ---
Files tested: 312778
Equal Error Rate (EER): 29.64%


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from sklearn.metrics import roc_curve

# -------- 1. Re-declare the Lightweight MLP Architecture --------
class SSLMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.network(x)


# -------- 2. EER Calculation Function --------
def calculate_eer(y_true, y_scores):
    """Calculates Equal Error Rate using ROC curve."""
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    frr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute(fpr - frr))]
    return eer * 100 # return as percentage


# -------- 3. Main Evaluation --------
def main():
    print("Loading evaluation data (this might take a moment)...")
    # Load the ~600k file CSV using the absolute path
    eval_df = pd.read_csv(r"C:\Users\adith\Desktop\Projects\Audio Deepfake\eval_df_ssl_features.csv")
    
    # Identify the feature columns dynamically
    feature_cols = [c for c in eval_df.columns if c.startswith("ssl_")]
    input_dim = len(feature_cols)
    
    # 1. Load the Scaler fitted on the training data
    print("Loading StandardScaler from training phase...")
    try:
        scaler = joblib.load(r"C:\Users\adith\Desktop\Projects\Audio Deepfake\ssl_scaler.pkl")
    except FileNotFoundError:
        print("ERROR: Cannot find 'ssl_scaler.pkl'. Please ensure it is in the specified directory.")
        return

    # 2. Load the Model
    print(f"Loading trained model with input dimension {input_dim}...")
    model = SSLMLP(input_dim)
    try:
        model.load_state_dict(torch.load(r"C:\Users\adith\Desktop\Projects\Audio Deepfake\ssl_mlp_weights.pt"))
    except FileNotFoundError:
        print("ERROR: Cannot find 'ssl_mlp_weights.pt'. Please ensure it is in the specified directory.")
        return
        
    model.eval() # Set to evaluation mode
    
    # 3. Separate into Clean (nocodec) and Compressed (everything else)
    clean_df = eval_df[eval_df['codec'] == 'nocodec']
    compressed_df = eval_df[eval_df['codec'] != 'nocodec']
    
    datasets = {"Clean (No Codec)": clean_df, "Compressed": compressed_df}
    
    for condition, df in datasets.items():
        if len(df) == 0:
            print(f"\n--- Condition: {condition} Audio ---")
            print("No files found for this condition.")
            continue
            
        X_eval = df[feature_cols].values
        y_eval = df['label'].values
        
        # Scale the features using the loaded scaler
        X_eval_scaled = scaler.transform(X_eval)
        
        # Convert to PyTorch tensors
        X_tensor = torch.tensor(X_eval_scaled, dtype=torch.float32)
        
        # 4. Get Predictions
        # We do this in batches manually so we don't blow up the RAM trying to process 600k files at once
        batch_size = 5000
        all_scores = []
        
        with torch.no_grad():
            for i in range(0, len(X_tensor), batch_size):
                batch = X_tensor[i:i+batch_size]
                logits = model(batch)
                scores = torch.sigmoid(logits).numpy().flatten()
                all_scores.extend(scores)
                
        all_scores = np.array(all_scores)
        
        # Calculate and print EER
        eer = calculate_eer(y_eval, all_scores)
        print(f"\n--- Condition: {condition} Audio ---")
        print(f"Files tested: {len(df)}")
        print(f"Equal Error Rate (EER): {eer:.2f}%")


if __name__ == "__main__":
    main()

Loading evaluation data (this might take a moment)...
Loading StandardScaler from training phase...
Loading trained model with input dimension 1536...

--- Condition: Clean (No Codec) Audio ---
Files tested: 44772
Equal Error Rate (EER): 18.75%

--- Condition: Compressed Audio ---
Files tested: 357161
Equal Error Rate (EER): 20.54%


In [3]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve
from tqdm import tqdm


# --- 1. Paths ---
# Use an absolute base directory instead of a dynamic relative path
BASE_DIR = r"C:\Users\adith\Desktop\Projects\Audio Deepfake"
SCALER_PATH = os.path.join(BASE_DIR, "fusion_academic_scaler.pkl")
WEIGHTS_PATH = os.path.join(BASE_DIR, "fusion_academic_mlp_weights.pt")
SSL_CSV = os.path.join(BASE_DIR, "eval_df_ssl_features.csv")
BIO_CSV = os.path.join(BASE_DIR, "eval_df_bio_academic.csv")


# --- 2. Architecture ---
class FusionMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x)


def calculate_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return ((fpr[idx] + fnr[idx]) / 2.0) * 100


# --- 3. RAM-Safe PyTorch Dataset ---
class DFEvalDataset(Dataset):
    def __init__(self, dataframe, scaler, feature_cols):
        # We store the dataframe reference, but we DO NOT call .values on the whole thing
        self.df = dataframe
        self.scaler = scaler
        self.feature_cols = feature_cols

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Only extract the specific row needed for this exact moment
        row = self.df.iloc[idx]
        features = row[self.feature_cols].values.astype(np.float32).reshape(1, -1)
        
        # Scale just this one row
        scaled_features = self.scaler.transform(features).flatten()
        label = np.float32(row["label"])
        
        return torch.tensor(scaled_features), torch.tensor(label)


def main():
    print("Loading Fusion Scaler...")
    try:
        scaler = joblib.load(SCALER_PATH)
    except FileNotFoundError:
        print(f"ERROR: Could not find scaler at {SCALER_PATH}")
        return

    model = FusionMLP(1550)
    try:
        model.load_state_dict(torch.load(WEIGHTS_PATH))
    except FileNotFoundError:
        print(f"ERROR: Could not find model weights at {WEIGHTS_PATH}")
        return
        
    model.eval()
    
    print("Loading SSL and Bio Evaluation datasets...")
    # Load chunks to save memory during merge
    ssl_df = pd.read_csv(SSL_CSV)
    bio_df = pd.read_csv(BIO_CSV).drop(columns=["label", "codec"])
    
    print("Fusing evaluation datasets...")
    eval_df = pd.merge(ssl_df, bio_df, on="filename", how="inner")
    
    # Delete massive raw dataframes from RAM to free up space immediately
    del ssl_df
    del bio_df
    
    print("Applying feature engineering...")
    eval_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    eval_df.dropna(inplace=True)
    eval_df["f0_cov"] = eval_df["stdevF0"] / (eval_df["meanF0"] + 1e-6)
    
    jsh_cols = ["j_local", "j_abs", "j_rap", "j_ppq5", "j_ddp", "s_local", "s_db", "s_apq3", "s_apq5", "s_apq11", "s_dda"]
    for col in jsh_cols:
        if col in eval_df.columns:
            eval_df[col] = np.log(np.clip(eval_df[col].astype(float), 1e-6, None))

    ssl_cols = [c for c in eval_df.columns if c.startswith("ssl_")]
    feature_cols = ssl_cols + ["meanF0", "f0_cov", "hnr"] + jsh_cols
    
    datasets = {
        "Clean (No Codec)": eval_df[eval_df['codec'] == 'nocodec'],
        "Compressed": eval_df[eval_df['codec'] != 'nocodec']
    }
    
    for condition, df in datasets.items():
        if len(df) == 0: continue
            
        print(f"\nEvaluating {condition} Audio ({len(df)} files)...")
        
        # Stream data row-by-row using DataLoader
        dataset = DFEvalDataset(df, scaler, feature_cols)
        loader = DataLoader(dataset, batch_size=2048, shuffle=False, num_workers=0)
         
        all_scores, all_labels = [], []
        
        with torch.no_grad():
            for batch_features, batch_labels in tqdm(loader, desc="Inference"):
                logits = model(batch_features)
                scores = torch.sigmoid(logits).numpy().flatten()
                
                all_scores.extend(scores)
                all_labels.extend(batch_labels.numpy())
                
        eer = calculate_eer(all_labels, all_scores)
        print(f"Equal Error Rate (EER): {eer:.2f}%")


if __name__ == "__main__":
    main()

Loading Fusion Scaler...
Loading SSL and Bio Evaluation datasets...
Fusing evaluation datasets...
Applying feature engineering...

Evaluating Clean (No Codec) Audio (44771 files)...


Inference: 100%|██████████| 22/22 [03:01<00:00,  8.25s/it]


Equal Error Rate (EER): 18.41%

Evaluating Compressed Audio (357150 files)...


Inference: 100%|██████████| 175/175 [24:51<00:00,  8.52s/it]


Equal Error Rate (EER): 20.25%
